In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

from test_attn_order_eval import ScoreOrderEval, summ

MYDTYPE = torch.bfloat16
MYDEVICE = 'cuda:0'


class SiLU(nn.SiLU):
    @property
    def output_multiplier(self) -> float:
        return 1.0
    # end
# end


@dataclass
class SimpleMLPConfig:
    dim_hidden: int
    dim_in: int
    dim_out: int
    bias: bool
    activation: nn.Module
# end

@dataclass
class SimpleTrainConfig:
    T: int
    L: int
# end


class SimpleMLP(nn.Module):
    def __init__(self, config_mlp):
        super().__init__()

        self.dim_hidden = config_mlp.dim_hidden
        self.dim_in = config_mlp.dim_in
        self.dim_out = config_mlp.dim_out
        self.bias = config_mlp.bias

        self.project_gate = nn.Linear(self.dim_in, self.dim_hidden, bias=self.bias, dtype=MYDTYPE)
        self.project_up = nn.Linear(self.dim_in, self.dim_hidden, bias=self.bias, dtype=MYDTYPE)
        self.project_down = nn.Linear(self.dim_hidden, self.dim_out, bias=self.bias, dtype=MYDTYPE)
        self.activation = config_mlp.activation
    # end

    def forward(self, x):
        return self.project_down(self.activation(self.project_gate(x)) * self.project_up(x))
    # end

    def device(self):
        return next(self.parameters()).device
    # end
# end


In [ ]:
def generate_mask_sequence(unmask):
    L, T = unmask.shape[0], unmask.shape[0]
    a = torch.full((L,), -1, dtype=torch.long)
    a[unmask] = torch.arange(T)
    b = a.view(1, -1) - torch.arange(T).view(-1, 1)
    return b
# end


def generate_y(unmask, l, h=5):
    a = torch.ones(l, dtype=torch.long, device=MYDEVICE)
    a[unmask] = torch.arange(l, device=MYDEVICE)
    b = a.view(1,-1) - torch.arange(l, device=MYDEVICE).view(-1,1)
    mask_current = (b <= h) & (b > 0)
    b[mask_current] = (h+1 - b[mask_current])
    b[~mask_current] = 0
    b = b/b[0].sum()

    # neg_inf = torch.finfo(b.dtype).min
    # b[~mask_current] = neg_inf
    return b
# end

def soft_rank_loss(pred: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    log_probs = F.log_softmax(pred, dim=-1)   # [B, L]
    loss = -(y * log_probs).sum(dim=-1)    # [B]
    return loss.mean()


In [3]:
torch.manual_seed(0)

config_mlp1 = SimpleMLPConfig(
    dim_in=3,
    dim_hidden=64,
    dim_out=64,
    bias=True,
    activation=SiLU()
)

config_mlp2 = SimpleMLPConfig(
    dim_in=64,
    dim_hidden=64,
    dim_out=64,
    bias=True,
    activation=SiLU()
)

config_mlp3 = SimpleMLPConfig(
    dim_in=64,
    dim_hidden=64,
    dim_out=1,
    bias=True,
    activation=SiLU()
)

model = nn.Sequential(
    SimpleMLP(config_mlp1),
    SimpleMLP(config_mlp2),
    SimpleMLP(config_mlp3)
).to('cuda:0')

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
config_train = SimpleTrainConfig(
    T=64,
    L=64
)


In [4]:
def train_model(model, optimizer, x, y, unmask, h=5):
    optimizer.zero_grad()
    mask_unmask = generate_mask_sequence(unmask) > 0

    losses = []
    for r in range(mask_unmask.shape[0]-h):
        output_r = model(x[r, mask_unmask[r]]).squeeze(-1)
        loss = soft_rank_loss(output_r, y[r,mask_unmask[r]])
        losses.append(loss)
        # print(loss)
    # end

    losses = torch.stack(losses).mean()
    losses.backward()
    optimizer.step()

    return losses.item()
# end


# x already merged
@ torch.no_grad()
def eval_model(model, x, unmask, id_batch, h=5):
    pred = model(x).squeeze(-1)
    eval = ScoreOrderEval(pred, unmask)
    print("[{}]  recall@5={}  pr_auc@5={}  ndcg@10={}".format(
        id_batch, summ(eval.recall_at_h(h)), summ(eval.pr_auc(h)), summ(eval.ndcg_at_h(2*h))))
    # end
# end

def main(model, optimizer, config_train=None, id_batch=0):
    T = config_train.T
    L = config_train.L

    id_blk = 0
    id_sample = 0
    folder_base = f'../stats_inf/{id_sample}'
    pos_root = 32
    h=5

    pos_base = pos_root + id_blk * L
    margin = torch.load(f'{folder_base}/margin_{pos_base}_{pos_base + L}.pt').to(dtype=MYDTYPE)
    conf = torch.load(f'{folder_base}/conf_{pos_base}_{pos_base + L}.pt').to(dtype=MYDTYPE)
    attn_last = torch.load(f'{folder_base}/attn_{pos_base}_{pos_base + L}.pt').squeeze(-2)[:,-1,:].to(dtype=MYDTYPE)
    unmask = torch.load(f'{folder_base}/unmask_{pos_base}_{pos_base + L}.pt').squeeze(-1) - pos_base

    x = torch.stack([attn_last, conf, margin], dim=-1)
    y = generate_y(unmask, L, h)
    

    loss = train_model(model, optimizer, x, y, unmask, h)
    print(f'[{id_batch}] loss: {loss}')
    eval_model(model, x, unmask, id_batch, h)
# end


In [5]:
for i in range(2000):
    main(model, optimizer, config_train, i)
# end

[0] loss: 3.352930784225464
[0]  recall@5=0.661 (n=59)  pr_auc@5=0.756 (n=58)  ndcg@10=0.837 (n=62)
[1] loss: 3.352771759033203
[1]  recall@5=0.654 (n=59)  pr_auc@5=0.648 (n=58)  ndcg@10=0.798 (n=62)
[2] loss: 3.3530189990997314
[2]  recall@5=0.654 (n=59)  pr_auc@5=0.634 (n=58)  ndcg@10=0.791 (n=62)
[3] loss: 3.3530189990997314
[3]  recall@5=0.651 (n=59)  pr_auc@5=0.651 (n=58)  ndcg@10=0.802 (n=62)
[4] loss: 3.3530189990997314
[4]  recall@5=0.668 (n=59)  pr_auc@5=0.724 (n=58)  ndcg@10=0.834 (n=62)
[5] loss: 3.3528249263763428
[5]  recall@5=0.678 (n=59)  pr_auc@5=0.750 (n=58)  ndcg@10=0.841 (n=62)
[6] loss: 3.352665901184082
[6]  recall@5=0.681 (n=59)  pr_auc@5=0.746 (n=58)  ndcg@10=0.841 (n=62)
[7] loss: 3.352665901184082
[7]  recall@5=0.678 (n=59)  pr_auc@5=0.743 (n=58)  ndcg@10=0.840 (n=62)
[8] loss: 3.352736711502075
[8]  recall@5=0.722 (n=59)  pr_auc@5=0.770 (n=58)  ndcg@10=0.835 (n=62)
[9] loss: 3.352489471435547
[9]  recall@5=0.681 (n=59)  pr_auc@5=0.790 (n=58)  ndcg@10=0.861 (n=

In [6]:
# T = config_train.T
# L = config_train.L

# id_batch = 0
# id_blk = 0
# folder_base = f'../stats_inf/{id_batch}'
# pos_root = 32
# h=5

# pos_base = pos_root + id_blk * L
# margin = torch.load(f'{folder_base}/margin_{pos_base}_{pos_base + L}.pt').to(dtype=MYDTYPE)
# conf = torch.load(f'{folder_base}/conf_{pos_base}_{pos_base + L}.pt').to(dtype=MYDTYPE)
# attn_last = torch.load(f'{folder_base}/attn_{pos_base}_{pos_base + L}.pt').squeeze(-2)[:,-1,:].to(dtype=MYDTYPE)
# unmask = torch.load(f'{folder_base}/unmask_{pos_base}_{pos_base + L}.pt').squeeze(-1) - pos_base

# x = torch.stack([attn_last, conf, margin], dim=-1)
# y = generate_y(unmask, L, h)

